In [1]:
# W12D3: Monitoring ML Models in Production
%pip install -q fastapi uvicorn scikit-learn pandas numpy pytest httpx ruff

print("W12D3 environment setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.8 MB/s eta 0:00:00
W12D3 environment setup complete!


In [2]:
# W12D3: Create ML prediction API

from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression

app = FastAPI(title="W12D3 Production ML Monitoring API")

# Sample training data
X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

model = LogisticRegression()
model.fit(X, y)

# Request format
class PredictionInput(BaseModel):
    value: float

# Health check
@app.get("/health")
def health():
    return {"status": "healthy", "model": "logistic-regression"}

# Prediction endpoint
@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    probability = float(model.predict_proba([[data.value]])[0][1])

    return {
        "input": data.value,
        "prediction": prediction,
        "positive_probability": round(probability, 4)
    }

print("W12D3 ML API created successfully!")
print("Endpoints: GET /health and POST /predict")


W12D3 ML API created successfully!
Endpoints: GET /health and POST /predict


In [3]:
# W12D3: Add API monitoring metrics

import time
from fastapi import Request

monitoring_stats = {
    "total_requests": 0,
    "failed_requests": 0,
    "total_latency_ms": 0.0
}

@app.middleware("http")
async def monitor_requests(request: Request, call_next):
    start_time = time.perf_counter()
    monitoring_stats["total_requests"] += 1

    try:
        response = await call_next(request)

        if response.status_code >= 400:
            monitoring_stats["failed_requests"] += 1

        return response

    except Exception:
        monitoring_stats["failed_requests"] += 1
        raise

    finally:
        elapsed_ms = (time.perf_counter() - start_time) * 1000
        monitoring_stats["total_latency_ms"] += elapsed_ms


@app.get("/monitoring")
def get_monitoring_metrics():
    total = monitoring_stats["total_requests"]

    return {
        **monitoring_stats,
        "average_latency_ms": round(
            monitoring_stats["total_latency_ms"] / total, 2
        ) if total else 0.0,
        "failure_rate_percent": round(
            monitoring_stats["failed_requests"] / total * 100, 2
        ) if total else 0.0
    }

print("Request monitoring middleware added!")
print("New endpoint: GET /monitoring")

Request monitoring middleware added!
New endpoint: GET /monitoring


In [4]:
# W12D3: Test API endpoints and collect monitoring metrics

from fastapi.testclient import TestClient
import json

client = TestClient(app)

# 1. Health check
health_response = client.get("/health")
print("Health check:", health_response.status_code)
print(health_response.json())

# 2. Test predictions
for value in [1, 3, 5]:
    response = client.post("/predict", json={"value": value})
    print(f"\nInput {value} | Status: {response.status_code}")
    print(response.json())

# 3. Collect monitoring metrics
metrics_response = client.get("/monitoring")
metrics = metrics_response.json()

print("\nProduction Monitoring Metrics:")
print(json.dumps(metrics, indent=2))

assert health_response.status_code == 200
assert metrics["total_requests"] >= 5


Health check: 200
{'status': 'healthy', 'model': 'logistic-regression'}

Input 1 | Status: 200
{'input': 1.0, 'prediction': 0, 'positive_probability': 0.1569}

Input 3 | Status: 200
{'input': 3.0, 'prediction': 1, 'positive_probability': 0.6365}

Input 5 | Status: 200
{'input': 5.0, 'prediction': 1, 'positive_probability': 0.9427}

Production Monitoring Metrics:
{
  "total_requests": 5,
  "failed_requests": 0,
  "total_latency_ms": 19.738369000037892,
  "average_latency_ms": 3.95,
  "failure_rate_percent": 0.0
}


In [5]:
# W12D3: Data drift detection and retraining trigger

import numpy as np

# Reference data used to train the model
reference_data = np.array([0, 1, 2, 3, 4, 5], dtype=float)

# Simulated recent production inputs
production_data = np.array([4.5, 5.0, 5.5, 6.0, 5.2, 5.8])

# Compare the mean of recent inputs with the training reference
reference_mean = float(np.mean(reference_data))
production_mean = float(np.mean(production_data))

mean_shift = abs(production_mean - reference_mean)
drift_threshold = 1.5

drift_detected = mean_shift > drift_threshold

# Monitoring decision
retraining_recommended = drift_detected

drift_report = {
    "reference_mean": round(reference_mean, 3),
    "production_mean": round(production_mean, 3),
    "absolute_mean_shift": round(mean_shift, 3),
    "drift_threshold": drift_threshold,
    "drift_detected": drift_detected,
    "retraining_recommended": retraining_recommended
}

print("Data Drift Report:")
print(drift_report)

if drift_detected:
    print("\nALERT: Feature distribution shifted. Investigate and evaluate retraining.")
else:
    print("\nNo drift alert based on the configured threshold.")

Data Drift Report:
{'reference_mean': 2.5, 'production_mean': 5.333, 'absolute_mean_shift': 2.833, 'drift_threshold': 1.5, 'drift_detected': True, 'retraining_recommended': True}

ALERT: Feature distribution shifted. Investigate and evaluate retraining.


In [6]:
# W12D3: Save monitoring outputs

import json
from pathlib import Path

project_dir = Path("/content/W12D3_Monitoring_ML_Models")
outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

# Save API health result
(outputs_dir / "health_output.json").write_text(
    json.dumps(health_response.json(), indent=2) + "\n"
)

# Save prediction result
prediction_response = client.post("/predict", json={"value": 4})
(outputs_dir / "prediction_output.json").write_text(
    json.dumps(prediction_response.json(), indent=2) + "\n"
)

# Save monitoring metrics
metrics = client.get("/monitoring").json()
(outputs_dir / "monitoring_metrics.json").write_text(
    json.dumps(metrics, indent=2) + "\n"
)

# Save data drift report
(outputs_dir / "drift_report.json").write_text(
    json.dumps(drift_report, indent=2) + "\n"
)

print("Saved output files:")
for file_path in sorted(outputs_dir.iterdir()):
    print("-", file_path.name)

print("\nMonitoring metrics:")
print(json.dumps(metrics, indent=2))

print("\nData drift report:")
print(json.dumps(drift_report, indent=2))

Saved output files:
- drift_report.json
- health_output.json
- monitoring_metrics.json
- prediction_output.json

Monitoring metrics:
{
  "total_requests": 7,
  "failed_requests": 0,
  "total_latency_ms": 24.434809000126734,
  "average_latency_ms": 3.49,
  "failure_rate_percent": 0.0
}

Data drift report:
{
  "reference_mean": 2.5,
  "production_mean": 5.333,
  "absolute_mean_shift": 2.833,
  "drift_threshold": 1.5,
  "drift_detected": true,
  "retraining_recommended": true
}


In [7]:
# W12D3: Create automated tests

tests_dir = project_dir / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction_endpoint():
    response = client.post("/predict", json={"value": 4})
    assert response.status_code == 200
    assert response.json()["prediction"] in [0, 1]
    assert "positive_probability" in response.json()


def test_monitoring_endpoint():
    response = client.get("/monitoring")
    assert response.status_code == 200
    data = response.json()
    assert "total_requests" in data
    assert "average_latency_ms" in data
    assert "failure_rate_percent" in data
'''

(tests_dir / "test_app.py").write_text(test_code.strip() + "\n")

print("Created tests/test_app.py")
print("Tests cover health, prediction, and monitoring endpoints.")

Created tests/test_app.py
Tests cover health, prediction, and monitoring endpoints.


In [8]:
# W12D3: Save the API implementation

app_code = '''
import time
from fastapi import FastAPI, Request
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression

app = FastAPI(title="W12D3 Production ML Monitoring API")

X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

model = LogisticRegression()
model.fit(X, y)

monitoring_stats = {
    "total_requests": 0,
    "failed_requests": 0,
    "total_latency_ms": 0.0
}

class PredictionInput(BaseModel):
    value: float

@app.middleware("http")
async def monitor_requests(request: Request, call_next):
    start = time.perf_counter()
    monitoring_stats["total_requests"] += 1
    try:
        response = await call_next(request)
        if response.status_code >= 400:
            monitoring_stats["failed_requests"] += 1
        return response
    except Exception:
        monitoring_stats["failed_requests"] += 1
        raise
    finally:
        elapsed = (time.perf_counter() - start) * 1000
        monitoring_stats["total_latency_ms"] += elapsed

@app.get("/health")
def health():
    return {"status": "healthy", "model": "logistic-regression"}

@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    probability = float(model.predict_proba([[data.value]])[0][1])
    return {
        "input": data.value,
        "prediction": prediction,
        "positive_probability": round(probability, 4)
    }

@app.get("/monitoring")
def get_monitoring_metrics():
    total = monitoring_stats["total_requests"]
    return {
        **monitoring_stats,
        "average_latency_ms": round(
            monitoring_stats["total_latency_ms"] / total, 2
        ) if total else 0.0,
        "failure_rate_percent": round(
            monitoring_stats["failed_requests"] / total * 100, 2
        ) if total else 0.0
    }
'''

(project_dir / "app.py").write_text(app_code.strip() + "\n")
print("Created app.py successfully!")

Created app.py successfully!


In [9]:
# W12D3: Run and save automated test results

import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

test_output = result.stdout + "\n" + result.stderr
print(test_output)

(outputs_dir / "test_results.txt").write_text(test_output)

if result.returncode != 0:
    raise RuntimeError("Tests failed. Please share the error output.")

print("SUCCESS: All tests passed!")
print("Saved: outputs/test_results.txt")

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D3_Monitoring_ML_Models
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_app.py::test_health_endpoint PASSED                           [ 33%]
tests/test_app.py::test_prediction_endpoint PASSED                       [ 66%]
tests/test_app.py::test_monitoring_endpoint PASSED                       [100%]

============================== 3 passed in 1.75s ===============================


SUCCESS: All tests passed!
Saved: outputs/test_results.txt


In [10]:
# W12D3: Create Docker configuration and dependencies

(project_dir / "requirements.txt").write_text(
    "fastapi\n"
    "uvicorn\n"
    "scikit-learn\n"
    "pydantic\n"
)

(project_dir / "requirements-dev.txt").write_text(
    "pytest\n"
    "httpx\n"
    "ruff\n"
)

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(project_dir / "Dockerfile").write_text(dockerfile.strip() + "\n")

gitignore = """
__pycache__/
*.py[cod]
.pytest_cache/
.ruff_cache/
.venv/
.env
.ipynb_checkpoints/
"""

(project_dir / ".gitignore").write_text(gitignore.strip() + "\n")

print("Created Dockerfile")
print("Created requirements.txt")
print("Created requirements-dev.txt")
print("Created .gitignore")

Created Dockerfile
Created requirements.txt
Created requirements-dev.txt
Created .gitignore


In [12]:
# W12D3: Production monitoring documentation

monitoring_text = """
# W12D3: Monitoring ML Models in Production

## 1. Metrics to Track
- API request count and error rate.
- Average and percentile response latency.
- CPU, memory, and container health.
- Input feature distributions and missing values.
- Prediction distributions and model confidence.
- Model quality when ground-truth labels become available.

## 2. Alerts
- API health check fails repeatedly.
- HTTP 5xx error rate exceeds 5% over a sustained window.
- P95 latency exceeds the agreed service-level target.
- Input data contains unexpected values or missing fields.
- Data drift exceeds the configured threshold.
- Model quality falls below the agreed performance threshold.

Thresholds are examples and should be calibrated to the service.

## 3. Data Drift
Compare recent production inputs with a representative training baseline.
Investigate shifts in feature distributions, input quality, and predictions.
The current project includes a simple mean-shift demonstration.

## 4. Retraining Triggers
- Validated model performance falls below its target.
- Persistent data drift is confirmed.
- Concept drift is observed using newly labelled outcomes.
- Sufficient new, representative training data is available.

Drift alone should trigger investigation, not automatic deployment.

## 5. Safe Retraining Process
1. Validate and clean new training data.
2. Train a candidate model.
3. Evaluate it against the existing model and baseline.
4. Run automated tests and check for regressions.
5. Deploy gradually after approval.
6. Monitor results and retain a rollback option.

## 6. Current Implementation
- FastAPI health and prediction endpoints.
- In-memory request, error, and latency metrics.
- Basic feature mean-shift drift check.
- Pytest API tests.
- Docker packaging and GitHub Actions build workflow.

## 7. Production Limitations
Metrics are currently in memory and reset when the process restarts.
A production service should export metrics to persistent monitoring
infrastructure, protect monitoring endpoints, and use statistical drift
tests appropriate to each feature and data type.
"""

(project_dir / "monitoring.md").write_text(
    monitoring_text.strip() + "\n"
)

print("Created monitoring.md")
print("Monitoring strategy documented successfully!")

Created monitoring.md
Monitoring strategy documented successfully!


In [14]:
# W12D3: Create GitHub Actions workflow

workflow_dir = project_dir / ".github" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)

workflow = """
name: W12D3 ML Monitoring CI/CD

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  lint-test-build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Lint Python files
        run: ruff check app.py tests

      - name: Run automated tests
        run: pytest -v

      - name: Build Docker image
        run: docker build -t w12d3-ml-monitoring:${{ github.sha }} .

      - name: Log in to Docker Hub
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Push Docker image
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: |
          docker tag w12d3-ml-monitoring:${{ github.sha }} ${{ secrets.DOCKERHUB_USERNAME }}/w12d3-ml-monitoring:${{ github.sha }}
          docker push ${{ secrets.DOCKERHUB_USERNAME }}/w12d3-ml-monitoring:${{ github.sha }}
"""

workflow_path = workflow_dir / "ci.yml"
workflow_path.write_text(workflow.strip() + "\n")

print("Created:", workflow_path)
print("Workflow file exists:", workflow_path.exists())
print("Stages: lint -> test -> Docker build -> optional registry push")

Created: /content/W12D3_Monitoring_ML_Models/.github/workflows/ci.yml
Workflow file exists: True
Stages: lint -> test -> Docker build -> optional registry push


In [15]:
# W12D3: Download each file separately

from google.colab import files
import time

files_to_download = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "README.md",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/monitoring_metrics.json",
    "outputs/drift_report.json",
    "outputs/test_results.txt",
]

missing = []

for name in files_to_download:
    path = project_dir / name

    if path.is_file():
        print("Downloading:", name)
        files.download(str(path))
        time.sleep(2)
    else:
        missing.append(name)
        print("MISSING:", name)

print("\nIndividual download requests completed.")

if missing:
    print("Please check these missing files:", missing)

Downloading: app.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: Dockerfile


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: outputs/test_results.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Individual download requests completed.
Please check these missing files: ['README.md']
